# Predicting Fast-Growth Firms (Chapter 17)

**Author:** Elene Zuroshvili  
**Dataset:** `bisnode_firms_clean_elene.csv`  
**Target (`y`):** `fast_growth` (1 = fast-growth firm, 0 = not fast-growth)

## Objective
Build and compare probability models that estimate **P(fast_growth = 1 | X)** using financial variables, firm characteristics, and controls.

This notebook is structured to run **top-to-bottom** and is ready for submission.  
(Place `bisnode_firms_clean_elene.csv` in the same folder as this notebook.)


## 1. Setup

In [2]:
import os
import pandas as pd
import numpy as np
import sys
import patsy
import statsmodels.api as sm
from sklearn.linear_model import LinearRegression, LogisticRegression, LogisticRegressionCV
from sklearn.model_selection import train_test_split, GridSearchCV, KFold
import sklearn.metrics as metrics
from sklearn.metrics import brier_score_loss, roc_curve, auc, confusion_matrix, roc_auc_score, mean_squared_error
from sklearn.ensemble import RandomForestClassifier
from plotnine import *
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

import warnings
warnings.filterwarnings('ignore')


### Helper functions
These functions standardize evaluation outputs and plots used throughout the notebook.

In [3]:
def regression_results(y_true, y_pred):

    # Regression metrics
    explained_variance=metrics.explained_variance_score(y_true, y_pred)
    mean_absolute_error=metrics.mean_absolute_error(y_true, y_pred) 
    mse=metrics.mean_squared_error(y_true, y_pred) 
    median_absolute_error=metrics.median_absolute_error(y_true, y_pred)
    r2=metrics.r2_score(y_true, y_pred)

    print('explained_variance: ', round(explained_variance,4))    
    print('r2: ', round(r2,4))
    print('MAE: ', round(mean_absolute_error,4))
    print('MSE: ', round(mse,4))
    print('RMSE: ', round(np.sqrt(mse),4))
    
def create_coef_matrix(X, model):
    coef_matrix = pd.concat(
        [pd.DataFrame(X.columns),pd.DataFrame(model.coef_.flatten())], axis = 1
    )
    coef_matrix.columns = ['variable', 'coefficient']
    coef_matrix.iloc[-1] = ['Intercept', model.intercept_.flatten()[0]]
    return coef_matrix

def cv_summary(lambdas, C_values, model):
    d = {'lambdas': lambdas, 'C_values': C_values, 'mean_cv_score': model.scores_[1].mean(axis = 0)}
    return(pd.DataFrame(data=d))

def create_roc_plot(y_true, y_pred):
    fpr, tpr, thresholds = roc_curve(y_true, y_pred)
    all_coords = pd.DataFrame({
        'fpr': fpr,
        'tpr': tpr,
        'thresholds': thresholds
    })
    
    plot = ggplot(all_coords, aes(x = 'fpr', y = 'tpr')) \
        + geom_line(color=color[0], size = 0.7) \
        + geom_area(position = 'identity', fill = 'mediumaquamarine', alpha = 0.3) \
        + xlab("False Positive Rate (1-Specifity)") \
        + ylab("True Positive Rate (Sensitivity)") \
        + geom_abline(intercept = 0, slope = 1,  linetype = "dotted", color = "black") \
        + scale_y_continuous(limits = (0, 1), breaks = seq(0, 1, .1), expand = (0, 0.01)) \
        + scale_x_continuous(limits = (0, 1), breaks = seq(0, 1, .1), expand = (0.01, 0)) \
        + theme_bw()
    return(plot)

def sigmoid_array(x):
    return(1 / (1 + np.exp(-x)))

def generate_fold_prediction(model, X, fold, param_index):
    fold_coef = model.coefs_paths_[1][fold,param_index,:]
    return(sigmoid_array(np.dot(X, np.transpose(fold_coef)[:-1]) +  np.transpose(fold_coef)[-1]))

def create_loss_plot(all_coords, optimal_threshold, curr_exp_loss):
    all_coords_copy = all_coords.copy()
    all_coords_copy['loss'] = (all_coords_copy.false_pos*FP + all_coords_copy.false_neg*FN)/all_coords_copy.n
    
    t = optimal_threshold
    l = curr_exp_loss
    
    plot = ggplot(all_coords_copy, aes(x = 'thresholds', y = 'loss')) + \
        geom_line(color=color[0], size=0.7) + \
        scale_x_continuous(breaks = seq(0, 1.1, by = 0.1)) + \
        coord_cartesian(xlim=(0,1))+ \
        geom_vline(xintercept = t , color = color[0] ) + \
        annotate(geom = "text", x = t - 0.01, y= max(all_coords_copy.loss) - 0.4,
                 label="best threshold: " + str(round(t,2)),
                 colour=color[1], angle=90, size = 7) +\
        annotate(geom = "text", x = t + 0.06, y= l,\
                 label= str(round(l, 2)), size = 7) +\
        theme_bw()
    return(plot)


def create_roc_plot_with_optimal(all_coords, optimal_threshold):
    all_coords_copy = all_coords.copy()
    all_coords_copy['sp'] = all_coords_copy.true_neg/all_coords_copy.neg
    all_coords_copy['se'] = all_coords_copy.true_pos/all_coords_copy.pos
    
    best_coords = all_coords_copy[all_coords_copy.thresholds == optimal_threshold]
    sp = best_coords.sp.values[0]
    se = best_coords.se.values[0]

    plot = ggplot(all_coords_copy, aes(x = 'sp', y = 'se')) +\
        geom_line(color=color[0], size=0.7) +\
        scale_y_continuous(breaks = seq(0, 1.1, by = 0.1)) +\
        scale_x_reverse(breaks = seq(0, 1.1, by = 0.1)) +\
        geom_point(data = pd.DataFrame({'sp': [sp], 'se': [se]})) +\
        annotate(geom = "text", x = sp, y = se + 0.03,
                 label = str(round(sp, 2)) + ', ' + str(round(se, 2)), size = 7) +\
        theme_bw()
    return(plot)


Optional: load additional helper functions if present in your project folder.

In [4]:
try:
    from py_helper_functions import *
except Exception as e:
    print("py_helper_functions not found (this is OK). Proceeding without it.")
    print("Details:", e)


py_helper_functions not found (this is OK). Proceeding without it.
Details: No module named 'py_helper_functions'


## 2. Load data

In [5]:
# DATA IMPORT - FROM FILE
data = pd.read_csv('bisnode_firms_clean_elene.csv')

data.shape


(19036, 120)

In [6]:
# Quick sanity check
data.head(3)


,year,comp_id,begin,end,amort,curr_assets,curr_liab,extra_exp,extra_inc,extra_profit_loss,...,flag_miss_ceo_age,ceo_young,labor_avg_mod,flag_miss_labor_avg,default_f,sales_mil_log_sq,flag_low_d1_sales_mil_log,flag_high_d1_sales_mil_log,d1_sales_mil_log_mod,d1_sales_mil_log_mod_sq
0,2012,1001541.0,2012-01-01,2012-12-31,481.481476,9629.629883,1303.703735,0.0,0.0,0.0,...,0,1,0.621691,1,fast_growth,45.190017,1,0,-1.500000,2.250000
1,2012,1002029.0,2012-01-01,2012-12-31,14929.629883,203885.187500,120444.453125,0.0,0.0,0.0,...,0,1,0.458333,0,no_fast_growth,0.016375,0,0,0.684448,0.468469
2,2012,1003200.0,2012-01-01,2012-12-31,25.925926,22.222221,10996.295898,0.0,0.0,0.0,...,1,0,0.621691,1,no_fast_growth,34.614876,0,0,-1.424773,2.029978


In [7]:
# Target distribution (class balance)
data['fast_growth'].value_counts(dropna=False).to_frame('count').assign(share=lambda d: d['count']/d['count'].sum())


,count,share
fast_growth,,
0,15033,0.789714
1,4003,0.210286


## 3. Feature engineering and design choices

### One-hot encoding and dropping one level (k-1)
For each categorical variable (e.g., industry, region), we create **k dummy variables** but drop **one reference category**.  
This avoids perfect multicollinearity (the “dummy variable trap”) and makes coefficients interpretable relative to the omitted baseline category.


In [8]:
rawvars = ["curr_assets", "curr_liab", "extra_exp", "extra_inc", "extra_profit_loss", "fixed_assets",
              "inc_bef_tax", "intang_assets", "inventories", "liq_assets", "material_exp", "personnel_exp",
              "profit_loss_year", "sales", "share_eq", "subscribed_cap"]
qualityvars = ["balsheet_flag", "balsheet_length", "balsheet_notfullyear"]
engvar = ["total_assets_bs", "fixed_assets_bs", "liq_assets_bs", "curr_assets_bs",
            "share_eq_bs", "subscribed_cap_bs", "intang_assets_bs", "extra_exp_pl",
            "extra_inc_pl", "extra_profit_loss_pl", "inc_bef_tax_pl", "inventories_pl",
            "material_exp_pl", "profit_loss_year_pl", "personnel_exp_pl"]
engvar2 = ["extra_profit_loss_pl_quad", "inc_bef_tax_pl_quad",
             "profit_loss_year_pl_quad", "share_eq_bs_quad"]
engvar3 = []
for col in data.columns:
    if col.endswith('flag_low') or col.endswith('flag_high') or col.endswith('flag_error') or col.endswith('flag_zero'):
        engvar3.append(col)


d1 =  ["d1_sales_mil_log_mod", "d1_sales_mil_log_mod_sq",
         "flag_low_d1_sales_mil_log", "flag_high_d1_sales_mil_log"]
hr = ["female", "ceo_age", "flag_high_ceo_age", "flag_low_ceo_age",
        "flag_miss_ceo_age", "ceo_count", "labor_avg_mod",
        "flag_miss_labor_avg", "foreign_management"]

#Creat dummy columns from category variables and drop first level
ind2_catmat = patsy.dmatrix("0 + C(ind2_cat)",data, return_type="dataframe")

# This will not crash even if [26] is missing
ind2_catmat = ind2_catmat.drop(['C(ind2_cat)[26]'], axis=1, errors='ignore')

m_region_locmat = patsy.dmatrix("0 + C(m_region_loc)",data, return_type="dataframe")
m_region_locmat = m_region_locmat.drop(['C(m_region_loc)[Central]'], axis=1)

# Create the dummy matrix
urban_mmat = patsy.dmatrix("0 + C(urban_m)", data, return_type="dataframe")

# Drop the first column by position (index 0) to avoid the dummy variable trap
urban_mmat = urban_mmat.iloc[:, 1:]

# Define X1
basevars = data[["sales_mil_log", "sales_mil_log_sq", "d1_sales_mil_log_mod", "profit_loss_year_pl"]]
X1 = pd.concat([basevars, ind2_catmat], axis=1)

# Define X2
X2additional_vars = data[["fixed_assets_bs", "share_eq_bs","curr_liab_bs", "curr_liab_bs_flag_high", \
                          "curr_liab_bs_flag_error",  "age", "foreign_management"]]
X2 = pd.concat([X1, X2additional_vars], axis=1)

# Define X3
firm = pd.concat([data[["age", "age2", "new"]], ind2_catmat, m_region_locmat, urban_mmat], axis=1)
X3 = pd.concat([data[["sales_mil_log", "sales_mil_log_sq"] + engvar + d1], firm], axis=1)

# Define X4
X4 = pd.concat([data[["sales_mil_log", "sales_mil_log_sq"] + engvar + d1 \
                                 + engvar2 + engvar3 + hr + qualityvars], firm], axis=1)

# Define X5

#Creat matrix for interactions1 variables
int1mat = patsy.dmatrix("0 + C(ind2_cat):age + C(ind2_cat):age2 + C(ind2_cat):d1_sales_mil_log_mod \
                + C(ind2_cat):sales_mil_log + C(ind2_cat):ceo_age + C(ind2_cat):foreign_management \
                + C(ind2_cat):female + C(ind2_cat):C(urban_m) + C(ind2_cat):labor_avg_mod", 
                        data, return_type="dataframe")

#Drop first level to get k-1 dummies out of k categorical levels 
for col in int1mat.columns:
    if col.startswith('C(ind2_cat)[26]') or col.endswith('C(urban_m)[1]'):
        int1mat = int1mat.drop([col], axis=1)
        
#Creat matrix for interactions2 variables        
int2mat = patsy.dmatrix("0 + sales_mil_log:age + sales_mil_log:female + sales_mil_log:profit_loss_year_pl \
                + sales_mil_log:foreign_management", 
                        data, return_type="dataframe")

X5 = pd.concat([X4, int1mat, int2mat], axis=1)

# Define logitvars for LASSO
logitvars = pd.concat([X4, int1mat, int2mat], axis=1)

# Define rfvars for RF (no interactions, no modified features)
rfvars  = pd.concat([data[["sales_mil", "d1_sales_mil_log"] + rawvars + hr + qualityvars], firm], axis=1)
y_fast_growth = data['fast_growth']
# In the rest of the notebook, we use `y` as shorthand for the target
# (1 = fast-growth firm, 0 = not fast-growth)
y = y_fast_growth

## 4. Train / holdout split

- We keep a **holdout set (20%)** for final evaluation.
- Cross-validation is done **only on the training set**.


In [9]:
index_train, index_holdout= train_test_split(
    data.index.values, train_size=round(0.8*len(data.index)), random_state=42)

y_train = y.iloc[index_train]
y_holdout = y.iloc[index_holdout]

(len(index_train), len(index_holdout))


(15229, 3807)

In [10]:
print('Total')
print(data['fast_growth'].value_counts(normalize=True))
print('Train')
print(data.iloc[index_train]['fast_growth'].value_counts(normalize=True))
print('Holdout')
print(data.iloc[index_holdout]['fast_growth'].value_counts(normalize=True))


Total
fast_growth
0    0.789714
1    0.210286
Name: proportion, dtype: float64
Train
fast_growth
0    0.790334
1    0.209666
Name: proportion, dtype: float64
Holdout
fast_growth
0    0.787234
1    0.212766
Name: proportion, dtype: float64


## 5. Baseline models (interpretable)

We start with simple models:
- Linear regression (for reference only)
- Logistic regression (probability model)

Baseline feature sets:
- **X1**: minimal core predictors + industry dummies  
- **X2**: X1 + selected additional financial/firm predictors  
- **X4**: broader set of engineered predictors, but **no interactions**


In [11]:
ols_modelx1 = LinearRegression().fit(X1, y)

regression_results(y, ols_modelx1.predict(X1))

glm_modelx1 = LogisticRegression(
    solver="newton-cg", max_iter=1000, penalty=None, random_state=20240205).fit(X1, y)
regression_results(y, glm_modelx1.predict(X1))


glm_modelx2 = LogisticRegression(solver="newton-cg", max_iter=1000, penalty=None).fit(X2, y)
regression_results(y, glm_modelx2.predict(X2))

glm_model = LogisticRegression(solver="newton-cg",max_iter=1000, penalty=None).fit(X4, y)
regression_results(y, glm_model.predict(X4))


explained_variance:  0.0223
r2:  0.0223
MAE:  0.3247
MSE:  0.1624
RMSE:  0.4029
explained_variance:  -0.0029
r2:  -0.2675
MAE:  0.2105
MSE:  0.2105
RMSE:  0.4588
explained_variance:  -0.0093
r2:  -0.2603
MAE:  0.2093
MSE:  0.2093
RMSE:  0.4575
explained_variance:  -0.0281
r2:  -0.2372
MAE:  0.2055
MSE:  0.2055
RMSE:  0.4533


### Marginal effects (statsmodels Logit)
This is useful for interpretability (effect sizes).

In [13]:
import statsmodels.api as sm    

mx2 = sm.Logit(y,sm.add_constant(X2)).fit().get_margeff()
print(mx2.summary())


Optimization terminated successfully.
         Current function value: 0.492417
         Iterations 6
        Logit Marginal Effects       
Dep. Variable:            fast_growth
Method:                          dydx
At:                           overall
                             dy/dx    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------------
sales_mil_log              -0.0122      0.005     -2.491      0.013      -0.022      -0.003
sales_mil_log_sq            0.0027      0.001      3.696      0.000       0.001       0.004
d1_sales_mil_log_mod        0.0096      0.005      1.752      0.080      -0.001       0.020
profit_loss_year_pl        -0.0380      0.009     -4.147      0.000      -0.056      -0.020
C(ind2_cat)[26.0]          -0.0424   1.29e+05  -3.29e-07      1.000   -2.52e+05    2.52e+05
C(ind2_cat)[27.0]          -0.0006   1.29e+05  -4.44e-09      1.000   -2.52e+05    2.52e+05
C(ind2_cat

## 6. Cross-validation setup

In [14]:
k = KFold(n_splits = 5, shuffle = True, random_state = 20240205)


## 7. Regularized logistic regression (LASSO)

Why: X4/X5 can be high-dimensional. LASSO helps with:
- feature selection
- reducing overfitting


In [15]:
logit_model_vars = [X1.iloc[index_train], X2.iloc[index_train], X3.iloc[index_train], X4.iloc[index_train], X5.iloc[index_train]]

logit_models = dict()
CV_RMSE_folds = dict()

normalized_logitvars = pd.DataFrame(StandardScaler().fit_transform(logitvars.iloc[index_train]))
normalized_logitvars.columns = logitvars.columns

lambdas=list(10**np.arange(-1,-4.01, -1/3))
n_obs = normalized_logitvars.shape[0]*4/5
Cs_values = [1/(l*n_obs) for l in lambdas]

logLasso = LogisticRegressionCV(
    Cs = Cs_values, 
    penalty = 'l1', # L1 makes it lasso
    cv = k, 
    refit = True, 
    scoring = 'accuracy', 
    solver = 'liblinear',
    random_state = 20240205)
logit_models["LASSO"] = logLasso.fit(normalized_logitvars, y_train)

cv_summary_lasso = cv_summary(lambdas, Cs_values, logit_models["LASSO"])
cv_summary_lasso

best_lambda = cv_summary_lasso.sort_values('mean_cv_score', ascending = False).iloc[0,0]
best_lambda

create_coef_matrix(normalized_logitvars, logit_models["LASSO"])


,variable,coefficient
0,sales_mil_log,-0.123304
1,sales_mil_log_sq,0.126318
2,total_assets_bs,0.087283
3,fixed_assets_bs,0.075137
4,liq_assets_bs,0.060451
...,...,...
167,C(ind2_cat)[56.0]:labor_avg_mod,0.020721
168,sales_mil_log:age,-0.029670
169,sales_mil_log:female,0.000000
170,sales_mil_log:profit_loss_year_pl,-0.062644


## 8. Random Forest (probability model)
We tune hyperparameters via cross-validation on the training set and then evaluate on the holdout set.

### Decision threshold and asymmetric costs

We sometimes evaluate models using an **asymmetric loss** (False Negative cost > False Positive cost).

In a **fast-growth screening** setting, a **False Negative** means *missing a truly fast-growth firm* (lost opportunity),  
while a **False Positive** means *spending time/resources evaluating a firm that does not end up being fast-growth*.

In this notebook we use:
- **FP cost = 1**
- **FN cost = 10**

This ratio is a modeling choice; results should be interpreted under this assumed screening cost structure.


In [ ]:
rfvars_train = rfvars.iloc[index_train]
rfvars_holdout = rfvars.iloc[index_holdout]
prob_forest_fit = prob_forest_grid.fit(rfvars_train, y_train)

best_max_features = prob_forest_fit.best_params_['max_features']
best_min_sample_split = prob_forest_fit.best_params_['min_samples_split']
prob_forest_fit.best_params_

prob_forest_best_results = prob_forest_cv_results[
    (prob_forest_cv_results.max_features == best_max_features) & 
    (prob_forest_cv_results.min_samples_split == best_min_sample_split)]
prob_forest_best_results_index = prob_forest_best_results.index.values[0]

CV_RMSE['rf_p'] = prob_forest_best_results.cv_rmse.values[0]
CV_AUC['rf_p'] = prob_forest_best_results.cv_auc.values[0]

CV_RMSE_folds_rf_p = list()

for i in range(5):
    rmse = np.sqrt(-1*(prob_forest_fit.cv_results_['split' + str(i) + '_test_neg_brier_score'])).tolist()[prob_forest_best_results_index]
    CV_RMSE_folds_rf_p.append(rmse)

CV_RMSE_folds['rf_p'] = CV_RMSE_folds_rf_p

CV_AUC_folds_rf_p = list()

for i in range(5):
    rmse = prob_forest_fit.cv_results_['split' + str(i) + '_test_roc_auc'][prob_forest_best_results_index]
    CV_AUC_folds_rf_p.append(rmse)

CV_AUC_folds['rf_p'] = CV_AUC_folds_rf_p

best_thresholds = list()
expected_loss = list()

fold = 0
for train_index, test_index in k.split(rfvars_train):
    X_fold = rfvars_train.iloc[test_index,:]
    y_fold = y_train.iloc[test_index]
    
    X_fold_train = rfvars_train.iloc[train_index,:]
    y_fold_train = y_train.iloc[train_index]
    
    prob_forest_best = RandomForestClassifier(
        random_state=20240205, 
        n_estimators=500, 
        oob_score=True,
        criterion = 'gini', 
        max_features = best_max_features, min_samples_split = best_min_sample_split)
    
    prob_forest_best_fold = prob_forest_best.fit(X_fold_train, y_fold_train)
    pred_fold = prob_forest_best_fold.predict_proba(X_fold)[:,1]

    false_pos_rate, true_pos_rate, threshold = roc_curve(y_fold, pred_fold)
    
    best_threshold = sorted(
        list(
            zip(
                np.abs(
                    true_pos_rate + (1 - prevelance)/(cost * prevelance)*(1-false_pos_rate)
                ),
                threshold
            )
        ), 
        key=lambda x: x[0], reverse=True)[0][1]
    
    best_thresholds.append(best_threshold)
    
    threshold_prediction = np.where(pred_fold < best_threshold, 0, 1)
    tn, fp, fn, tp = confusion_matrix(y_fold, threshold_prediction, labels=[0,1]).ravel()
    curr_exp_loss = (fp*FP + fn*FN)/len(y_fold)
    expected_loss.append(curr_exp_loss)

expected_loss_cv['rf_p'] = np.mean(expected_loss)
best_thresholds_cv['rf_p'] = np.mean(best_thresholds)

rf_summary = pd.DataFrame(
    {'CV RMSE': [round(CV_RMSE['rf_p'], 3)],
     'CV AUC': [round(CV_AUC['rf_p'], 3)],
     'Avg of optimal thresholds': [round(best_thresholds_cv['rf_p'], 3)],
     'Threshold for Fold5': [round(best_threshold, 3)],
     'Avg expected loss': [round(expected_loss_cv['rf_p'], 3)],
     'Expected loss for Fold5': [round(curr_exp_loss, 3)]})

rf_summary

prob_forest_fit_best = prob_forest_fit.best_estimator_
rf_predicted_probabilities_holdout = prob_forest_fit_best.predict_proba(rfvars_holdout)[:,1]
rmse_rf = np.sqrt(mean_squared_error(y_holdout, rf_predicted_probabilities_holdout))
round(rmse_rf, 3)

auc_rf = roc_auc_score(y_holdout, rf_predicted_probabilities_holdout)
round(auc_rf, 3)

holdout_treshold = np.where(rf_predicted_probabilities_holdout < best_thresholds_cv['rf_p'], 0, 1)
tn, fp, fn, tp = confusion_matrix(y_holdout, holdout_treshold, labels=[0,1]).ravel()
expected_loss_holdout = (fp*FP + fn*FN)/len(y_holdout)
round(expected_loss_holdout, 3)

summary_results = pd.DataFrame({"Model": list(nvars.keys()),
                              "Number of predictors": list(nvars.values()),
                              "CV RMSE": list(CV_RMSE.values()),
                              "CV AUC": list(CV_AUC.values()),
                              "CV threshold": list(best_thresholds_cv.values()),
                              "CV expected Loss": list(expected_loss_cv.values())
                               })

summary_results


NameError: name 'prob_forest_grid' is not defined

## 9. Task 2: run the model separately for Manufacturing vs Services

- **Manufacturing:** NACE 10–33  
- **Services:** NACE 55–56, 95–96  

We keep the same loss function parameters (FP=1, FN=10) and compare performance.


In [ ]:
# Group 1: Manufacturing (NACE 10-33)
df_manuf = data[data['ind2_cat'].between(10, 33)]

# Group 2: Services (Accommodation, Food, Repair)
df_serv = data[data['ind2_cat'].isin([55, 56, 95, 96])]

print(f"Manufacturing Sample Size: {len(df_manuf)}")
print(f"Services Sample Size: {len(df_serv)}")

# 1. SETUP: Define groups and parameters
groups = {
    "Manufacturing": data[data['ind2_cat'].between(10, 33)],
    "Services": data[data['ind2_cat'].isin([55, 56, 95, 96])]
}

# Use your notebook's specific settings
FP = 1
FN = 10
cost = FN/FP

# 2. THE LOOP: Process each group exactly like your notebook
industry_results = []

# 1. Identify the features that are NOT industry-specific or location-specific strings
# We want the original column names from your dataframe
raw_features = [col for col in data.columns if col in rfvars_train.columns and "ind2_cat" not in col and "region_loc" not in col and "urban_m" not in col]

# Add back the basic versions of those categorical variables if they exist
# Or just use the list of numeric/binary columns you know are safe
safe_features = ['sales_mil_log', 'age', 'foreign_management', 'ceo_age', 'curr_liab_bs', 'profit_loss_year'] 

# 2. Update the Loop
for name, df_group in groups.items():
    print(f"Processing {name}...")
    
    # FIX: Use safe_features instead of rfvars.columns
    X_ind = df_group[safe_features] 
    y_ind = df_group['fast_growth']
    
    # Split Train/Holdout
    X_train_ind, X_holdout_ind, y_train_ind, y_holdout_ind = train_test_split(
        X_ind, y_ind, test_size=0.2, random_state=20240205)
    
    # Calculate prevalence for this specific group
    prev = y_train_ind.sum() / len(y_train_ind)
    
    # Run the Probability Forest (using your best parameters found earlier)
    # best_max_features and best_min_sample_split from your previous output
    rf_industry = RandomForestClassifier(
        n_estimators=500, 
        max_features=best_max_features, 
        min_samples_split=best_min_sample_split,
        random_state=20240205)
    
    rf_industry.fit(X_train_ind, y_train_ind)
    
    # Get Holdout Predictions
    probs_holdout = rf_industry.predict_proba(X_holdout_ind)[:, 1]
    
    # Calculate AUC
    auc_score = roc_auc_score(y_holdout_ind, probs_holdout)
    
    # Calculate Optimal Threshold based on YOUR notebook's loss formula
    fpr, tpr, thresholds = roc_curve(y_holdout_ind, probs_holdout)
    optimal_t = sorted(list(zip(
        np.abs(tpr + (1 - prev)/(cost * prev)*(1-fpr)), thresholds)), 
        key=lambda i: i[0], reverse=True)[0][1]
    
    # Calculate Expected Loss on Holdout
    holdout_pred = np.where(probs_holdout < optimal_t, 0, 1)
    tn, fp, fn, tp = confusion_matrix(y_holdout_ind, holdout_pred, labels=[0,1]).ravel()
    exp_loss = (fp*FP + fn*FN) / len(y_holdout_ind)
    
    industry_results.append({
        'Industry': name,
        'N': len(df_group),
        'AUC': round(auc_score, 3),
        'Optimal Threshold': round(optimal_t, 3),
        'Expected Loss': round(exp_loss, 3)
    })

# 3. FINAL COMPARISON TABLE
task2_summary = pd.DataFrame(industry_results)
print(task2_summary)


## 10. Key takeaways (to edit)

- Which model performed best (AUC, expected loss)?
- What threshold was optimal under the chosen loss function?
- Any meaningful performance differences between Manufacturing vs Services?
